In [1]:
from pdb import set_trace as st
from pprint import pprint
from collections import defaultdict

import msgspec
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import re 
from multiprocessing import Pool, cpu_count
from json_repair import repair_json

from sutime import SUTime
sutime = SUTime(mark_time_ranges=True, include_range=True)

from python_heideltime import Heideltime
heideltime_parser = Heideltime()
heideltime_parser.set_document_type("NEWS")

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Registering annotator sutime with class edu.stanford.nlp.time.TimeAnnotator
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator pos
[main] INFO edu.stanford.nlp.tagger.maxent.MaxentTagger - Loading POS tagger from edu/stanford/nlp/models/pos-tagger/english-left3words-distsim.tagger ... done [1.0 sec].
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
[main] INFO edu.stanford.nlp.sequences.SeqClassifierFlags - sutime.includeRange=true
[main] INFO edu.stanford.nlp.sequences.SeqClassifierFlags - Unknown property: |sutime.includeRange|
[main] INFO edu.stanford.nlp.sequences.SeqClassifierFlags - sutime.language=english
[main] INFO edu.stanford.nlp.sequ

In [ ]:
def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()
    
    output = []

    with open(file_path, "rb") as file:
        data = file.read()
        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp

# Read the corpus data

In [24]:
DATA_PATH = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_temporal_v2.jsonl")
train_temporal_jsonl = read_json(DATA_PATH, jsonl=True)

The file is of type: <class 'list'>
The file contains 8060 items.


In [30]:
new_train_jsonl = []

for ix, line in tqdm(enumerate(train_temporal_jsonl), total=len(train_temporal_jsonl)):
    
    positive_passages, negative_passages = line["positive_passages"], line["negative_passages"]
    
    temp = {}
    temp["query_id"] = line["query_id"]
    temp["query"] = line["query"]
    temp["positive_passages"] = []
    temp["negative_passages"] = []

    for item in positive_passages:
        pos_docid = item["docid"]
        pos_text = item["text"]
        temp["positive_passages"].append(
            {"docid": pos_docid, "text": pos_text}
        )

    for item in negative_passages:
        neg_docid = item["docid"]
        neg_text = item["text"]
        temp["negative_passages"].append(
            {"docid": neg_docid, "text": neg_text}
        )
        
    new_train_jsonl.append(temp)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8060/8060 [00:00<00:00, 264629.00it/s]


In [51]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]

In [52]:
print(TemporalAnnotation.model_json_schema())

{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 

# VLLM generation

In [ ]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]


In [633]:
print(TemporalAnnotation.model_json_schema())

{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 

In [639]:
import os

# export VLLM_USE_V1=1
# export TOKENIZERS_PARALLELISM=0
os.environ["VLLM_USE_V1"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "0"


from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams
from vllm.distributed import cleanup_dist_env_and_memory

guided_decoding_params = GuidedDecodingParams(
    json=TemporalAnnotation.model_json_schema(),
)

sampling_params = SamplingParams(
    max_tokens=32768, 
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0,
    guided_decoding=guided_decoding_params
)

llm = LLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
    generation_config="auto",
    max_model_len=32768,  # Limit context window
    max_num_seqs=4,  # Limit batch size
    gpu_memory_utilization=0.95,
    disable_cascade_attn=True,  # Avoid gibberish output due to batch inference
    seed=42,
)

INFO 10-02 09:50:18 [__init__.py:241] Automatically detected platform cuda.
INFO 10-02 09:50:45 [utils.py:326] non-default args: {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'seed': 42, 'max_model_len': 32768, 'enable_prefix_caching': True, 'disable_cascade_attn': True, 'gpu_memory_utilization': 0.95, 'max_num_seqs': 4, 'disable_log_stats': True, 'enable_chunked_prefill': True}
INFO 10-02 09:51:43 [__init__.py:711] Resolved architecture: Qwen3ForCausalLM
INFO 10-02 09:51:43 [__init__.py:1750] Using max model len 32768
INFO 10-02 09:51:54 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 10-02 09:51:55 [llm_engine.py:222] Initializing a V0 LLM engine (v0.10.1.1) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, do

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 10-02 09:52:57 [default_loader.py:262] Loading weights took 40.82 seconds
INFO 10-02 09:52:59 [model_runner.py:1112] Model loading took 7.6065 GiB and 43.945656 seconds
INFO 10-02 09:53:07 [worker.py:295] Memory profiling takes 6.72 seconds
INFO 10-02 09:53:07 [worker.py:295] the current vLLM instance can use total_gpu_memory (44.53GiB) x gpu_memory_utilization (0.95) = 42.30GiB
INFO 10-02 09:53:07 [worker.py:295] model weights take 7.61GiB; non_torch_memory takes 0.08GiB; PyTorch activation peak memory takes 0.16GiB; the rest of the memory reserved for KV Cache is 34.45GiB.
INFO 10-02 09:53:08 [executor_base.py:114] # cuda blocks: 15680, # CPU blocks: 1820
INFO 10-02 09:53:08 [executor_base.py:119] Maximum concurrency for 32768 tokens per request: 7.66x
INFO 10-02 09:53:10 [model_runner.py:1383] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' i

Capturing CUDA graph shapes:   0%|          | 0/3 [00:00<?, ?it/s]

INFO 10-02 09:53:17 [model_runner.py:1535] Graph capturing finished in 7 secs, took 0.15 GiB
INFO 10-02 09:53:17 [llm_engine.py:417] init engine (profile, create kv cache, warmup model) took 17.72 seconds
INFO 10-02 09:53:26 [llm.py:298] Supported_tasks: ['generate']


In [ ]:
prompt_template = """You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME's output for your reference. We also provide you with previously annotated positive and negative samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.
Your final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.
Your temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. 
You must follow the definitions and instructions below.

* Allen relations and descriptions:
- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?
- After: Did ‘Event A’ occur after ‘Event B’ without any overlap between the two events?
- Meets: Did ‘Event A’ end in the same time as ‘Event B’ began? Answer True or False.
- MetBy: Did ‘Event B’ end in the same time as ‘Event A’ began? Answer True or False.
- Overlaps: Did ‘Event A’ begin before ‘Event B’ and end before ‘Event B’ ended, with some overlap between the two events?
- OverlappedBy: Did ‘Event B’ begin before ‘Event A’ and end before ‘Event A’ ended, with some overlap between the two events?
- Starts: ‘Event A’ begin in the same time as ‘Event B’, but end before ‘Event B’ ended?
- StartedBy: Did ‘Event B’ begin in the same time as ‘Event A’, but end before ‘Event A’ ended?
- During: Did ‘Event A’ begin after ‘Event B’ began and end before ‘Event B’ ended, being entirely contained within ‘Event B’?
- Contains: Did ‘Event A’ begin before ‘Event B’ began and end after ‘Event B’ ended, entirely containing ‘Event B’?
- Finishes: Did ‘Event A’ begin after ‘Event B’ began and end in the same time as ‘Event B’?
- FinishedBy: Did ‘Event B’ begin after ‘Event A’ began and end in the same time as ‘Event A’?
- Equals: Did ‘Event A’ begin in the same time as ‘Event B’ and end in the same time as ‘Event B’?
- Empty: a special case for TemporalAnswer where there is no temporal expression.

* Temporal signals (examples of what can appear in queries or passages): before, prior to, until, after, following, since, in, on, as of, for duration, during, while, when, from...to..., between, by, up to, first, last, around, as soon as, as long as, for, over, all through, throughout, etc.

* Taxonomy of "positive_passages" and "negative_passages" with examples that you should generate:
- Explicit temporal constraints (i.e., always have clear temporal expressions that can be anchored to a specific datetime): "who won the state of Texas in 2008?"; "what kind of government does Iran have after 1979?".
- Implicit temporal constraints (i.e., events that cannot be anchored to specific datetime. The rule of thumb is: if there is date or month or year in the question, like "before the **2004** general election", it is not implicit temporal): "who was the president after JFK died?"; "what team did Michael Jordan play for after the Bulls?".
- TemporalAnswer (i.e., Questions that inquire the datetime of an event instead of a general question with temporal constraints, usually starts with "when" or "what/which + day/date/month/year". There is no temporal expression in it.): "what year did the Knicks win the championship?"; "when was the United Nations founded?".

* Instructions:
1. You must output one or multiple valid JSONs, delimited by a newline, strictly following this pydantic schema:
{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'positive_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Positive Passages', 'type': 'array'}, 'negative_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Negative Passages', 'type': 'array'}}, 'required': ['query_id', 'query', 'temporal', 'positive_passages', 'negative_passages'], 'title': 'TemporalAnnotation', 'type': 'object'}.

2. "query_id": must be assigned with the provided "query_id".

3. “query":
- A query is a provided sentence that may contain one or many temporal expressions.
- Please note that SUTIME only provides explicit temporal expressions for the query and they are not perfect; therefore, you must double-check them and further detect additional explicit and implicit temporal expressions such as events.
- You must extract all events that are anchored to specific datetime (e.g., “2000 FA Cup Final”, “the 2007 election”, etc.).
- The "temporal" field should be extracted as written from the query's text and they must be concise, such as: "in April, 1906", "2 July 2010", "2004 general election", etc.

4. "positive_passages":
- Natural, QA-style questions with diverse phrasing that seek information from the query. Each question must contain exactly one extractable temporal expression, logically align with one of the query's temporal expressions, yet be diverse in Allen relations.
- Based on the query's temporal expressions, you must generate "positive_passages" that cover all "TemporalQueryType", including 'Explicit', 'Implicit', and 'TemporalAnswer', when possible:
    - You MUST prioritise generating questions with explicit temporal constraints, like "in 2010", "from 2010 to 2015", etc. They must logically align with the query’s temporal expression(s), such as being equals, overlapping with, or being contained within the query’s temporal expression(s). If the query has multiple temporal expressions, the generated questions must logically align with at least one of them.
    - You should also prioritise generating questions with implicit temporal constraints, such as "after EVENT", "before EVENT". These must logically align with the query’s temporal expression(s).
    - For both explicit and implicit "TemporalQueryType" questions, you must not use phrases like what/which date/day/month/year/time or when etc., that inquire about time. You must not confuse this with TemporalAnswer questions.
    - You can also generate "TemporalAnswer" (asking for datetime/duration/time-range of an EVENT, e.g., "When did EVENT happen?", "What time did he arrive?"). For this type of question, you must not add any temporal expression and you must set: "TemporalQueryType": "TemporalAnswer", "allen_relation": "Empty", and "temporal": []. Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- "temporal" field: must extract all exact text spans of the temporal expressions that exist in the generated question (not normalized or paraphrased). Regarding explicit and implicit "TemporalQueryType", they must be concise temporal expressions as written, such as "after July 2010", "from 2012 to 2014", that are not just normalized dates. For instance, instead of "In 1906 he moved with his family to a farm", prefer the concise "In 1906" and keep prepositions or context words that anchor the time, e.g., 'from', 'in', 'after', etc. Regarding "TemporalAnswer" passages, the "temporal" field must be an empty list.
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- “docid”: must remain the same as provided.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality positive passages, prioritizing quality over quantity.

5. "negative_passages": follow the same format as "positive_passages", but represent contexts that are temporally mismatched or semantically irrelevant to the query. Based on the query's temporal expressions, you must generate "negative_passages" that cover all "TemporalQueryType", including "Explicit", "Implicit", and "TemporalAnswer", when possible. They must be hard, temporally-confused, yet diverse in Allen relations questions that cover all following cases:
- Case 1: Questions with temporal expressions that mismatch with the query’s temporal expression(s): 
    - If the query specifies a span (e.g., 2005–2007), use an interval that falls completely outside it (e.g., 2008, before 2005, after 2007).
    - Adjacent years or ranges are valid negatives (e.g., query = 2010, negative = 2009).
    - Use shifted but non-overlapping intervals (e.g., query = 2010, negative = 2012–2014).
    - Include misleading implicit cues (e.g., “shortly after 2011” vs. query “in 2010”).
    - You may reuse the same passage from "positive_passages" but replace its temporal expressions with mismatched ones.
- Case 2: Questions with same temporal expression but irrelevant event/entity
    - Questions with overlapping or identical temporal expressions but targeting a different subject.
    - Example: query = “Ronaldo’s career in 2010” vs. negative = “Messi’s career in 2010”.
    - This ensures negatives are temporally aligned but semantically irrelevant.
- Case 3: "TemporalAnswer"-type questions that are either:
    - Irrelevant to the query.
    - Or ask for non-existent temporal information in the query context.
    - Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality negative passages, prioritizing quality over quantity.  
- Do not create trivial negatives (e.g., completely unrelated random text).
- Ensure no accidental overlap with valid facts in the document.

6. "temporal_query_type": Must be "Explicit", "Implicit", or "TemporalAnswer". If the temporal contains a clear date/month/year, it is "Explicit", NOT "Implicit". If the passage asks for which date/month/year of an event, it is "TemporalAnswer".
   
7. Final output: Only output valid JSON(s). Do not explain, add comments, or include extra text, since your output will be parsed automatically.

### Demonstration 1
Input:
docid: 2465
query_id: 0
query: "On 2 July 2010 , after helping Setúbal avoid top-flight relegation , Barbosa was released by Porto , signing a three-year contract with S.C . Braga."
SUTIME's output: [{'timex-value': '2010-07-02', 'start': 3, 'end': 14, 'text': '2 July 2010', 'type': 'DATE', 'value': '2010-07-02'}, {'timex-value': 'P3Y', 'start': 111, 'end': 121, 'text': 'three-year', 'type': 'DURATION', 'value': 'P3Y'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?.
Example negative passages: "Hélder Barbosa played for which team from 2002 to 2009?", "Hélder Barbosa played for which team from 2006 to 2009?".

Output:
{"query_id":0,"query":"On 2 July 2010, after helping Setúbal avoid top-flight relegation, Barbosa was released by Porto, signing a three-year contract with S.C. Braga.","temporal":["2 July 2010","three-year contract"],"positive_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2010 to 2013?","temporal":["from 2010 to 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for on 2 July 2010?","temporal":["2 July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa join after leaving Porto in July 2010?","temporal":["after leaving Porto in July 2010"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa sign a three-year contract with?","temporal":["three-year contract"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"After being released by Porto, which team did Hélder Barbosa sign a contract with?","temporal":["After being released by Porto"],"allen_relation":"MetBy","temporal_query_type":"Implicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between July 2010 and July 2013?","temporal":["between July 2010 and July 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"When did Hélder Barbosa sign a contract with S.C. Braga.?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2465,"text":"How long did Hélder Barbosa's contract with S.C. Braga last?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2002 to 2009?","temporal":["from 2002 to 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between 2006 and 2009?","temporal":["between 2006 and 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for after 2014?","temporal":["after 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for during 2008?","temporal":["during 2008"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for prior to 2010?","temporal":["prior to 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Ronaldo play for in July 2010?","temporal":["in July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 1

### Demonstration 2
Input:
docid: 2466
query_id: 1
query: "Rarely used in the first months , he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish ."
SUTIME's output: [{'timex-value': 'PXM', 'start': 15, 'end': 31, 'text': 'the first months', 'type': 'DURATION', 'value': 'PXM'}, {'timex-value': '2011-01', 'start': 78, 'end': 90, 'text': 'January 2011', 'type': 'DATE', 'value': '2011-01'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?"
Example negative passages: Hélder Barbosa played for which team from 2002 to 2009?; Hélder Barbosa played for which team from 2006 to 2009?.

{"query_id":1,"query":"Rarely used in the first months, he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish .","temporal":["after the January 2011 departure of Matheus"],"positive_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for between January 2011 and December 2011?","temporal":["between January 2011 and December 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for as of January 2011?","temporal":["as of January 2011"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after Matheus departed in January 2011?","temporal":["after Matheus departed in January 2011"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa contribute goals to during the 2011 season?","temporal":["during the 2011 season"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for throughout 2011?","temporal":["throughout 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"When did Hélder Barbosa start getting more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2466,"text":"When did Hélder Barbosa begin gaining more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for before 2010?","temporal":["before 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after 2013?","temporal":["after 2013"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for in 2007?","temporal":["in 2007"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for prior to joining Braga?","temporal":["prior to joining Braga"],"allen_relation":"Before","temporal_query_type":"Implicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for during 2014?","temporal":["during 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"After the January 2011 departure of Matheus, did Ronaldo get more playing time?","temporal":["After the January 2011 departure of Matheus"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 2"""

import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")

def normalize_and_split_string_by_punctuation(text):    
    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)
    
    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)    

    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"
    
    # Remove : and ;
    punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    return [x.strip() for x in re.findall(punct_regex, text)]



In [656]:
a = {"query_id":2,"query":"According to some sources , Disney worried about the rising criminality of the city . A neighboring family had two adolescent children involved in a car barn robbery , and Disney feared that crime would taint his own children . In 1906 he moved with his family to a farm near Marceline , Missouri . Disney and his family settled there in April , 1906 . On March 5 , he bought a farm . Its previous owner William E . Crane had died in November , 1905 . Crane was a veteran of the American Civil War and his house predated the foundation of Marceline . He bought the farm for $3,000 or $75 per acre . On April 3 , Disney bought an adjoining tract of about from Cranes widow . He paid an additional $450 .","positive_passages":[{"docid":2756,"text":"What was the residence of Elias Disney from 1906 to 1910?"},{"docid":2756,"text":"What was the residence of Elias Disney in October 1906?"}],"negative_passages":[{"docid":2756,"text":"What was the residence of Elias Disney from 1911 to 1912?"},{"docid":2756,"text":"What was the residence of Elias Disney from 1911 to 1913?"}]}
text = a["query"]
query_id = a["query_id"]
docid = a["positive_passages"][0]["docid"]
positive_passages = a["positive_passages"]
negative_passages = a["negative_passages"]

positive_passages = [x['text'] for x in positive_passages]
negative_passages = [x['text'] for x in negative_passages]

query_list = normalize_and_split_string_by_punctuation(text)
final_query_list = []
final_sutime_list = []

messages = []

final_query_list = []
final_sutime_list = []

buffer = []

for q in query_list:
    parsed = sutime.parse(q)

    if len(parsed) == 0:
        # No temporal expression → keep merging
        buffer.append(q)
    else:
        # Check if merging with buffer creates more temporal expressions
        if buffer:
            merged = " ".join(buffer + [q])
            if len(sutime.parse(merged)) > 1:
                temp = " ".join(buffer)
                final_query_list.append(temp)
                final_sutime_list.append(sutime.parse(temp))
                buffer = [q]
                continue
        buffer.append(q)

# Flush leftover
if buffer:
    merged = " ".join(buffer)
    temp = sutime.parse(merged)
    if len(temp) == 1:
        final_query_list.append(merged)
        final_sutime_list.append(temp)

for q, sutime_output in zip(final_query_list, final_sutime_list):
    # sutime_output = sutime.parse(q.lower()) 
    
    if not (len(sutime_output) > 0 and len(word_count_regex.findall(q)) > 5):
        continue

    for t in sutime_output:
        s, e = t["start"], t["end"]
        t["value"] = q[s:e]

    content = f"Input:\ndocid: {docid}\nquery_id: {query_id}\nquery: {q}\nSUTIME's output: {sutime_output}\nExample positive passages: {','.join(positive_passages)}\nExample negative passages:{';'.join(negative_passages)}\nOutput:"
    
    messages.append([
        {"role":"system", "content":prompt_template},
        {"role":"user", "content":{content},}
    ])

In [657]:
print(query_list)
print(final_query_list, len(final_sutime_list), [len(sutime.parse(x.lower())) for x in final_query_list])

['According to some sources, Disney worried about the rising criminality of the city.', 'A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children.', 'In 1906 he moved with his family to a farm near Marceline, Missouri.', 'Disney and his family settled there in April, 1906.', 'On March 5, he bought a farm.', 'Its previous owner William E. Crane had died in November, 1905.', 'Crane was a veteran of the American Civil War and his house predated the foundation of Marceline.', 'He bought the farm for $3, 000 or $75 per acre.', 'On April 3, Disney bought an adjoining tract of about from Cranes widow.', 'He paid an additional $450.']
['According to some sources, Disney worried about the rising criminality of the city. A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children. In 1906 he moved with his family to a farm near Marceline

In [111]:
output = llm.chat(
    messages=messages, 
    sampling_params=sampling_params,
    # chat_template_kwargs={"enable_thinking": True},
)

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

In [115]:
output

[RequestOutput(request_id=31, prompt=None, prompt_token_ids=[151644, 8948, 198, 2610, 525, 264, 35915, 21223, 6203, 369, 1995, 56370, 13, 1205, 3410, 498, 448, 264, 3239, 11, 6718, 504, 264, 1293, 2197, 11, 429, 1231, 6644, 825, 476, 1657, 35915, 23393, 323, 279, 12159, 328, 1381, 5660, 594, 2550, 369, 697, 5785, 13, 1205, 1083, 3410, 498, 448, 8597, 62851, 6785, 323, 8225, 10469, 369, 279, 4024, 1293, 2197, 26, 498, 1231, 25978, 1105, 476, 990, 1105, 438, 15057, 1172, 421, 807, 525, 5234, 80949, 9760, 311, 279, 1482, 3239, 13, 4615, 2618, 374, 311, 6923, 1550, 22092, 6785, 46769, 323, 8225, 46769, 369, 35915, 12872, 533, 6832, 624, 7771, 1590, 2550, 525, 62851, 4718, 82, 320, 2152, 69592, 8, 429, 25470, 1795, 279, 3897, 4718, 10802, 624, 7771, 35915, 32207, 1969, 387, 803, 23560, 323, 14720, 1091, 328, 1381, 5660, 476, 1260, 26802, 1462, 11, 448, 1909, 57255, 13403, 13, 715, 2610, 1969, 1795, 279, 17473, 323, 11221, 3685, 382, 9, 20060, 4300, 323, 27787, 510, 12, 13235, 25, 14568, 336

In [ ]:
temporal_answer_list = [
    # Point in time
    "what day",
    "what time",
    "what date",
    "what month",
    "what year",
    "which day",
    "which time",
    "which date",
    "which month",
    "which year",
    "when did",
    "when was",
    "when were",
    "at what time",
    "at what date",
    "at what year",
    "on what day",
    "on what date",
    "on what year",

    # Duration / span
    "how long",
    "how many days",
    "how many weeks",
    "how many months",
    "how many years",
    "for how long",
    "over what period",
    "during what years",
    "during which year",
    "for what duration",
    "in what year range",
    "between what years",

    # Frequency / recurrence
    "how often",
    "how frequently",

    # Relative temporals
    "since when",
    "until when",
    "from when",
    "from what year",
    "from what date",
    "to what year",
    "to what date",
    "up to when",
    "as of when",
    "by what year",
    "by when",
    "around when",
    "at what age",
]


def fix_implicit_temporal(passage, sutime):
    """
    Adjust temporal_query_type if 'implicit' but tagger detects explicit datetime.
    """
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Implicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    if len(sutime.parse(passage["temporal"][0].lower())) > 0:
        passage["temporal_query_type"] = TemporalQueryType.Explicit
        
    lower_passage = passage["text"].lower()
    
    # Make sure that the
    implicit_temporal = []
    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            implicit_temporal.append(passage["text"][start:start+len(t)])    
    
    if not implicit_temporal:
        return {}
    
    passage["temporal"] = implicit_temporal
    return passage


def fix_temporal_answer(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.TemporalAnswer:
        return passage

    lower_passage = passage['text'].lower()
    if any(word in lower_passage for word in temporal_answer_list):
        for word in temporal_answer_list:
            if word in lower_passage:
                lower_passage = lower_passage.replace(word, "")
        
    # Ensure that it passes the SUTIME tagger.
    if len(sutime.parse(lower_passage)) > 0:
        return {}
    else:
        passage['temporal'] = []
        passage['allen_relation'] = AllenRelation.Empty
    # print(passage)
    return passage


def fix_explicit_temporal(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Explicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    lower_passage = passage["text"].lower()
    temporal = []

    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            temporal.append(passage["text"][start:start+len(t)])

    if not temporal:
        return {}

    passage["temporal"] = temporal
    return passage


def validate_passages(passages, sutime):
    """
    Validate and filter passages.
    """
    valid = []
    
    for p in passages:
        p = fix_implicit_temporal(p, sutime)
        p = fix_explicit_temporal(p, sutime)
        p = fix_temporal_answer(p, sutime)
        
        if p:
            valid.append(p)
    return valid


def validate_temporal(js, max_temporal_expressions=3):
    lower_query = js["query"].lower()
    temporal = []
    
    # Ensure that the temporal is extracted as written from the original query
    for t in js["temporal"]:
        start = lower_query.find(t.lower())
        if start != -1:
            temporal.append(js["query"][start:start+len(t)])

    if len(temporal) == 0 or len(temporal) > max_temporal_expressions:
        # print(temporal)
        return []
    return temporal


temporal_jsonl = []
temp = []
for sample in output:
    sample = sample.outputs[0].text # Get the generated text
    list_of_raw_js = []
    query_set = set()
    
    for x in sample.split("\n"): # Split by newline to get multiple JSONs if any
        parts = x.split(',{"query_id"')
        if len(parts) > 0:
            json_strings = [parts[0]] + [',{"query_id"' + p for p in parts[1:]]
            list_of_raw_js.extend(json_strings)
        else:
            list_of_raw_js.append(x)
    
    for raw_js in list_of_raw_js:
        raw_js = raw_js.strip()
        if raw_js == "":
            continue
        try:
            js = TemporalAnnotation.model_validate_json(repair_json(raw_js, ensure_ascii=False)).model_dump()
            
            if len(js["temporal"]) == 0 or len(js["positive_passages"]) == 0 or len(js["negative_passages"]) == 0 or js["query"] in query_set:
                continue
            else:
                print(js["query"])
                js["temporal"] = validate_temporal(js)
                
                if len(js["temporal"]) == 0:
                    continue
                
                # Validate passages
                js["positive_passages"] = validate_passages(
                    js["positive_passages"], sutime
                )
                
                if len(js["positive_passages"]) == 0:
                    continue
                
                js["negative_passages"] = validate_passages(
                    js["negative_passages"], sutime
                )
                
                if len(js["negative_passages"]) == 0:
                    continue
                
                temp.append(js)
                query_set.add(js["query"])
                # positive = defaultdict(int)
                # negative = defaultdict(int)
                # allen_relation = defaultdict(int)

                # for j in js["positive_passages"]:
                #     positive[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                # for j in js["negative_passages"]:
                #     negative[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                
                # print(positive)
                # pprint(positive)
                # print(positive)
                # pprint(negative)

        except Exception as e:
            print("Error:", e)
            print("Offending JSON:", raw_js)
            continue
temporal_jsonl.extend(temp)

According to some sources, Disney worried about the rising criminality of the city. A neighboring family had two adolescent children involved in a car barn robbery, and Disney feared that crime would taint his own children. In 1906 he moved with his family to a farm near Marceline, Missouri.
Disney and his family settled there in April, 1906.
On March 5, he bought a farm.
Its previous owner William E. Crane had died in November, 1905. Crane was a veteran of the American Civil War and his house predated the foundation of Marceline. He bought the farm for $3, 000 or $75 per acre.
On April 3, Disney bought an adjoining tract of about from Cranes widow. He paid an additional $450.


In [118]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v3/test.jsonl", temporal_jsonl, jsonl=True)

The file contains 5 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v3/test.jsonl


In [ ]:
del llm
cleanup_dist_env_and_memory()

# Post-processing

In [5]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 11798 items.


In [6]:
# temporal_jsonl
positive = defaultdict(int)
negative = defaultdict(int)
allen_relation = defaultdict(int)
for x in temp_jsonl:
    for i in x["positive_passages"]:
        positive[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
    for i in x["negative_passages"]:
        negative[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
pprint(positive)
pprint(negative)

defaultdict(<class 'int'>,
            {'Explicit': 51193,
             'Implicit': 974,
             'TemporalAnswer': 13262})
defaultdict(<class 'int'>,
            {'Explicit': 60474,
             'Implicit': 281,
             'TemporalAnswer': 292})


## Check for relative temporal expressions in the queries

In [102]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

# new_temp_jsonl = []

count = 0

for x in temp_jsonl:

    if "now" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "now")
        count += 1
        # break
    if "today" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "today")
        count += 1
        # break
    if "current" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "current")
        count += 1 
        # break
    if "yesterday" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "yesterday")
        count += 1     
        # break
        
    # if count != 0 and (any([i["temporal_query_type"] == "Explicit" for i in x["positive_passages"]]) or any([i["temporal_query_type"] == "Explicit" for i in x["negative_passages"]])):
    #     continue
    
#     new_temp_jsonl.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)

The file is of type: <class 'list'>
The file contains 11693 items.


## Check for limited positive/negative passages

In [51]:
# new_temp_jsonl = []
for x in temp_jsonl:
    if len(x["positive_passages"]) <= 1:
        print("pos", x["query_id"])
        count += 1
        continue
    if len(x["negative_passages"]) <= 1:
        print("nega", x["query_id"])
        count += 1
        continue
#     new_temp_jsonl.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)

## Check for positive/negative passages that only contain a reference year (2025)

In [55]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

count = 0
for x in temp_jsonl:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if all(t):
        print(x["query_id"])
        print(t)
        count += 1
print(count)   

The file is of type: <class 'list'>
The file contains 11704 items.
0


## Ensure that either positive and negative passages must contain at least one explicit/implicit temporal

In [83]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", jsonl=True)

count_pos = 0
count_neg = 0
count_both = 0
count = 0
query_set = set()
final_query_list = []

for x in temp_jsonl:
    check_pos = False

    for i in x["positive_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_pos = True
            break 

    if check_pos == False:
        # print("pos", x["query_id"], x["query"][:10])
        count_pos += 1
        query_set.add(x["query_id"])

    check_neg = False    
    for i in x["negative_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_neg = True
            break

    if check_neg == False:
        count_neg += 1
        # print("neg", x["query_id"], x["query"][:10])
        query_set.add(x["query_id"])
        
    # if check_neg == True and check_pos == True:
    #     count += 1
    #     count_both += 1
    #     query_set.add(x["query_id"])
    #     final_query_list.append(x)
    
    if not check_neg and not check_pos:
        count_both += 1
        print(x["query_id"], x["query"][:10])

# print(count)

The file is of type: <class 'list'>
The file contains 11693 items.


In [84]:
print(count, count_pos, count_neg, count_both)

0 26 4 0


# Time-sensitive QA

In [3]:
from wikimapper import WikiMapper

# wiki_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/enwiki-20211220-pages-articles.jsonl", jsonl=True)
mapper = WikiMapper("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/index_enwiki-20220820.db")

easy_train_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/easy/train.easy.json", jsonl=True)

hard_train_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/hard/train.hard.json", jsonl=True)

hard_dev_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/hard/dev.hard.json", jsonl=True)

hard_test_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/hard/test.hard.json", jsonl=True)

hard_jsonl = hard_train_json + hard_dev_json + hard_test_json

# print(mapper.title_to_id("Carl_Eric_Almgren")) # Q5040099
# print(mapper.url_to_id("/wiki/Carl_Eric_Almgren#P39#1")) # None

The file is of type: <class 'list'>
The file contains 14308 items.
The file is of type: <class 'list'>
The file contains 14681 items.
The file is of type: <class 'list'>
The file contains 3087 items.
The file is of type: <class 'list'>
The file contains 3078 items.


## Create the corpus

### Split the large JSONL into chunk size JSONLs

In [ ]:
# split_folder_path = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/split")

# # temp_jsonl = []

# # with open("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/enwiki-20211220-pages-articles.jsonl") as f:
# #     for ix, line in tqdm(enumerate(f)):
# #         if ix < 10000000:
# #             continue
# #         # if ix >= 10000000:
# #         #     break
# #         temp_jsonl.append(decoder.decode(line))
# print(len(temp_jsonl))
# write_json(split_folder_path / "10.jsonl", temp_jsonl, jsonl=True)
# Finally all converted into parquets

In [ ]:
corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/temp.parquet")
corpus_parquet["title"] = corpus_parquet["title"].apply(lambda x: x[1:-1])

In [42]:
corpus_parquet.head()["text"].values[3]

'As a studio musician he worked with Willi Resetarits and Maria Bill. He has also worked with Savina Yannatou, Lucía Pulido, Kurt Ostbahn and Krzysztof Dobrek. Tom Lord lists 27 shots 1993-2012 in the field of Jazz.  Besides Preinfalk also acted as a theater composer. For the Wiener Volkstheater he wrote the music to "Peer Gynt", "Job" and "You stay with me", each directed by Michael Sturminger. Also he arranged Jewish songs with lyrics from 1952 murdered under Stalin poets ("Moscow 52", "Bukovina III").  2011 Preinfalk was appointed as a university professor of saxophone at the Kunstuniversität Graz.'

## Extract the wiki articles

In [265]:
corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/corpus.parquet")
train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/hard/train.hard.json", jsonl=True)
dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/hard/dev.hard.jsonl", jsonl=True)
test_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/hard/test.hard.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 14660 items.
The file is of type: <class 'list'>
The file contains 3087 items.
The file is of type: <class 'list'>
The file contains 3078 items.


In [ ]:
wiki_url_regex = re.compile(r"([^#]+)")

def process(x):
    return mapper.url_to_id(re.sub(r"amp;", "", wiki_url_regex.match(x['idx'])[0]))

train_id = []
dev_id = []
test_id = []

if __name__ == '__main__':
    with Pool(16) as pool:  # use all available cores
        train_id = list(tqdm(
            pool.imap(process, train_jsonl), 
            total=len(train_jsonl)
        ))
        
    with Pool(16) as pool:  # use all available cores
        dev_id = list(tqdm(
            pool.imap(process, dev_jsonl), 
            total=len(dev_jsonl)
        ))
        
    with Pool(16) as pool:  # use all available cores
        test_id = list(tqdm(
            pool.imap(process, test_jsonl), 
            total=len(test_jsonl)
        ))
        

  0%|          | 0/14660 [00:00<?, ?it/s]

  0%|          | 0/3087 [00:00<?, ?it/s]

  0%|          | 0/3078 [00:00<?, ?it/s]

### Filter out hard set documents from the wikipedia corpus (22453 entries in total)

In [270]:
result_df = pd.DataFrame()
hard_json_id_set = set(train_id + dev_id + test_id)

for i in tqdm(range(11)):
    temp_jsonl_df = pd.read_parquet(f"/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/flashrag/split/{i}.parquet")

    temp_df = temp_jsonl_df[
        temp_jsonl_df['id'].isin(hard_json_id_set) & 
        temp_jsonl_df['id'].notna()
    ]
    
    result_df = pd.concat([result_df, temp_df])
    print(len(result_df))

result_df = result_df.reset_index()
result_df["title"] = result_df["title"].apply(lambda x: x[2:-2])
result_df.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/corpus.parquet")
corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/corpus.parquet")

  0%|          | 0/11 [00:00<?, ?it/s]

1288
2707
4589
7101
9308
11501
13125
15035
19459
21572
22453


### Save the splits into parquets for later use

In [327]:
for iy, (i, j) in tqdm(enumerate(zip(test_jsonl, test_id))):
    temp = corpus_parquet[corpus_parquet["id"].values == j]
    if len(temp) == 0:
        print(iy)
    # i["docid"] = []
    i["wiki_id"] = temp["id"].values[0] if len(temp["id"].values) != 0 else None
    answer = i["targets"]
    
    # for a in answer:
    #     for ix, t in enumerate(temp["text"]):
    #         # Strip all punctuations for more correct search 
    #         t = re.sub(r'[^\w\s]','',t, re.UNICODE)  
    #         if a in t:
    #             i["docid"].append(temp.index[ix])
df = pd.DataFrame(test_jsonl)
df["unanswerable"] = df["targets"].apply(lambda x: x == [''])
df.to_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test.parquet")

0it [00:00, ?it/s]

2257
2258
2259


In [3]:
df_train = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/train.parquet")
df_dev = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/dev.parquet")
df_test = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test.parquet")

print(len(df_train[df_train["wiki_id"].isna()]), len(df_train[df_train["unanswerable"].values == True]))
print(len(df_dev[df_dev["wiki_id"].isna()]), len(df_dev[df_dev["unanswerable"].values == True]))
print(len(df_test[df_test["wiki_id"].isna()]), len(df_test[df_test["unanswerable"].values == True]))

9 2149
0 413
3 465


### Extract the paragraphs for each question in the train split

In [178]:
def search_paragraphs_index(recon_start_end_list, answer_start_end, recon_context, targets, paragraphs):
    """Return the index of the paragraph span that contains the answer span."""
    potential_ix = []
    for ix, (s, e) in enumerate(recon_start_end_list):
        if targets in paragraphs[ix]['text']:
            potential_ix.append(ix)
        if answer_start_end[0] >= s and answer_start_end[1] <= e and targets in paragraphs[ix]['text']:
            return ix
    
    potential_start_end_list = [recon_start_end_list[x] for x in potential_ix]
    for ix, (s, e) in enumerate(potential_start_end_list):
        if answer_start_end[0] >= s:
            return ix
    return -1

def format_paragraph(paragraphs, art_title, idx, dedup_titles):
    par_title = paragraphs[idx]["title"]
    if dedup_titles and art_title == par_title:
        title_str = art_title
    else:
        title_str = f"{art_title} - {par_title}"
    return f"{title_str}: {paragraphs[idx]['text']}"

def extract_paragraphs_with_answers(paragraphs, answer_starts, answer_ends, targets, dedup_titles=True):
    """
    Given Wikipedia-style paragraphs and answer spans (start/end indices),
    return a list of disambiguated paragraph strings that include both
    article title and section title.
    
    Parameters
    ----------
    paragraphs : list[dict]
        Each dict must have keys: "title", "text".
    answer_starts : list[int]
        List of answer start indices (aligned with reconstructed context).
    answer_ends : list[int]
        List of answer end indices.
    dedup_titles : bool, default=True
        If True, collapses "Title - Title" into "Title".
    
    Returns
    -------
    list[str] : formatted paragraphs with titles and text
    """
    
    title_set = set()
    start_end_list = []
    recon_context = ""
    final_paragraph_list = []

    # Build context + paragraph spans
    for par in paragraphs:
        title = par["title"]
        text = par["text"].strip()

        # Title case
        if len(title_set) == 0: 
            recon_context += title.strip()
            title_set.add(title)

        # Add article/section title once
        if title not in title_set:
            recon_context += " " + title.strip() + " . "
            title_set.add(title)

        start_idx = len(recon_context)
        recon_context += " " + text.strip() + " "
        end_idx = len(recon_context)

        start_end_list.append((start_idx, end_idx))
    recon_context = recon_context.replace("  ", " ")

    # Match answers back to paragraph(s)
    for t, s, e in zip(targets, answer_starts, answer_ends):
        ix = search_paragraphs_index(start_end_list, (s, e), recon_context, t, paragraphs)
        art_title = paragraphs[0]["title"]

        if ix != -1:
            final_paragraph_list.append(format_paragraph(paragraphs, art_title, ix, dedup_titles))
            continue

        # fallback: search in recon_context
        t_lower = t.lower()
        for ix, (_, end) in enumerate(start_end_list):
            if t_lower in recon_context[:end].lower():
                nearby_idxs = range(max(0, ix - 1), min(len(paragraphs), ix + 1))
                combined = " ".join(
                    format_paragraph(paragraphs, art_title, j, dedup_titles)
                    for j in nearby_idxs
                )
                final_paragraph_list.append(combined)
                break

    return final_paragraph_list


In [626]:
output_jsonl = []
count = 0 

for i in range(len(df_train)):
    sample = df_train.iloc[i]
    
    if sample["unanswerable"]:
        continue
    
    targets = sample['targets'].tolist()
    start, end = sample["from"].tolist(), sample["end"].tolist()
    paragraphs = sample['paragraphs'].tolist()
    # print(len(sample['context']), sample['context'])

    result = extract_paragraphs_with_answers(paragraphs, start, end, targets)
    
    if len(result) == 0:
        # Final resort
        temp = normalize_and_split_string_by_punctuation(sample["context"])
        for ix, chunk in enumerate(temp):
            if targets[0] in chunk:
                start = max(0, ix - 5)
                end = min(len(temp), ix + 5)
                result = temp[start:end]
                break
        if len(result) == 0:
            print(i, sample['targets'][0], sample['targets'][0] in sample['context'])
            count += 1
            
    for res in result:
        output_jsonl.append({
            "query_id": i,
            "query": res,
            "positive_passages": [{"docid":sample["wiki_id"], "text": sample["question"]}]
        })

7851 A.S . Livorno Calcio True
9060 U.S . Congress True
9921 Warner Bros.-Seven Arts True
13791 U.S . House of Representatives True


In [627]:
output_jsonl[0]

{'query_id': 0,
 'query': 'Knox Cunningham - Parliament:  In the 1955 general election , Cunningham was chosen as the new Ulster Unionist MP for South Antrim . He was a delegate to the Council of Europe and Western European Union Parliamentary Assembly from 1956 to 1959 . He also served as Parliamentary Private Secretary to Jocelyn Simon , Financial Secretary to the Treasury , from 1958 . In 1959 he was made a Queens Counsel .',
 'positive_passages': [{'docid': 'Q13529889',
   'text': 'Which position did Knox Cunningham hold before Apr 1956?'}]}

### Merge duplicate passages together and accumulate the questions of the same passage

In [ ]:
temp = defaultdict(list)
new_train_jsonl = []

for i in output_jsonl:
    temp[i["query"]].append(i["positive_passages"][0])

for ix, (k, v) in tqdm(enumerate(temp.items())):
    if v[0]["docid"] is None:
        continue
    new_train_jsonl.append({
        "query_id": ix,
        "query": k,
        "positive_passages": v,
    })
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/train.jsonl", output_jsonl, jsonl=True)

In [ ]:
new_train_jsonl[0]

{'query_id': 0,
 'query': 'Knox Cunningham - Parliament:  In the 1955 general election , Cunningham was chosen as the new Ulster Unionist MP for South Antrim . He was a delegate to the Council of Europe and Western European Union Parliamentary Assembly from 1956 to 1959 . He also served as Parliamentary Private Secretary to Jocelyn Simon , Financial Secretary to the Treasury , from 1958 . In 1959 he was made a Queens Counsel .',
 'positive_passages': [{'docid': 'Q13529889',
   'text': 'Which position did Knox Cunningham hold before Apr 1956?'},
  {'docid': 'Q13529889',
   'text': 'Which position did Knox Cunningham hold between Jun 1956 and Sep 1956?'},
  {'docid': 'Q13529889',
   'text': 'Which position did Knox Cunningham hold between Jul 1957 and Jul 1958?'},
  {'docid': 'Q13529889',
   'text': 'Which position did Knox Cunningham hold between Jul 1957 and Jul 1958?'}]}

## VLLM generation

In [673]:
from pydantic import BaseModel
from typing import List
from enum import Enum

# Allen relations
class AllenRelation(str, Enum):
    Before = "Before"
    After = "After"
    Meets = "Meets"
    MetBy = "MetBy"
    Overlaps = "Overlaps"
    OverlappedBy = "OverlappedBy"
    Starts = "Starts"
    StartedBy = "StartedBy"
    During = "During"
    Contains = "Contains"
    Finishes = "Finishes"
    FinishedBy = "FinishedBy"
    Equals = "Equals"
    Empty = "Empty"

# Temporal type
class TemporalQueryType(str, Enum):
    Explicit = "Explicit"
    Implicit = "Implicit"
    # Ordinal = "Ordinal"
    TemporalAnswer = "TemporalAnswer"

# # Reasoning level
# class ReasoningLevel(str, Enum):
#     L2 = "L2"  # Time-Event
#     L3 = "L3"  # Event-Event

# Passage model
class Passage(BaseModel):
    docid:int # The corresponding docid in the corpus
    text:str
    temporal:List[str]
    temporal_query_type:TemporalQueryType
    allen_relation:AllenRelation
    # reasoning_level:ReasoningLevel

# Full JSON schema
class TemporalAnnotation(BaseModel):
    query_id:int
    query:str
    temporal:List[str]
    positive_passages:List[Passage]
    negative_passages:List[Passage]

print(TemporalAnnotation.model_json_schema())

{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 

In [684]:
prompt_template = """You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME's output for your reference. We also provide you with previously annotated positive samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.
Your final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.
Your temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. 
You must follow the definitions and instructions below.

* Allen relations and descriptions:
- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?
- After: Did ‘Event A’ occur after ‘Event B’ without any overlap between the two events?
- Meets: Did ‘Event A’ end in the same time as ‘Event B’ began? Answer True or False.
- MetBy: Did ‘Event B’ end in the same time as ‘Event A’ began? Answer True or False.
- Overlaps: Did ‘Event A’ begin before ‘Event B’ and end before ‘Event B’ ended, with some overlap between the two events?
- OverlappedBy: Did ‘Event B’ begin before ‘Event A’ and end before ‘Event A’ ended, with some overlap between the two events?
- Starts: ‘Event A’ begin in the same time as ‘Event B’, but end before ‘Event B’ ended?
- StartedBy: Did ‘Event B’ begin in the same time as ‘Event A’, but end before ‘Event A’ ended?
- During: Did ‘Event A’ begin after ‘Event B’ began and end before ‘Event B’ ended, being entirely contained within ‘Event B’?
- Contains: Did ‘Event A’ begin before ‘Event B’ began and end after ‘Event B’ ended, entirely containing ‘Event B’?
- Finishes: Did ‘Event A’ begin after ‘Event B’ began and end in the same time as ‘Event B’?
- FinishedBy: Did ‘Event B’ begin after ‘Event A’ began and end in the same time as ‘Event A’?
- Equals: Did ‘Event A’ begin in the same time as ‘Event B’ and end in the same time as ‘Event B’?
- Empty: a special case for TemporalAnswer where there is no temporal expression.

* Temporal signals (examples of what can appear in queries or passages): before, prior to, until, after, following, since, in, on, as of, for duration, during, while, when, from...to..., between, by, up to, first, last, around, as soon as, as long as, for, over, all through, throughout, etc.

* Taxonomy of "positive_passages" and "negative_passages" with examples that you should generate:
- Explicit temporal constraints (i.e., always have clear temporal expressions that can be anchored to a specific datetime): "who won the state of Texas in 2008?"; "what kind of government does Iran have after 1979?".
- Implicit temporal constraints (i.e., events that cannot be anchored to specific datetime. The rule of thumb is: if there is date or month or year in the question, like "before the **2004** general election", it is not implicit temporal): "who was the president after JFK died?"; "what team did Michael Jordan play for after the Bulls?".
- TemporalAnswer (i.e., Questions that inquire the datetime of an event instead of a general question with temporal constraints, usually starts with "when" or "what/which + day/date/month/year". There is no temporal expression in it.): "what year did the Knicks win the championship?"; "when was the United Nations founded?".

* Instructions:
1. You must output one or multiple valid JSONs, delimited by a newline, strictly following this pydantic schema:
{'$defs': {'AllenRelation': {'enum': ['Before', 'After', 'Meets', 'MetBy', 'Overlaps', 'OverlappedBy', 'Starts', 'StartedBy', 'During', 'Contains', 'Finishes', 'FinishedBy', 'Equals', 'Empty'], 'title': 'AllenRelation', 'type': 'string'}, 'Passage': {'properties': {'docid': {'title': 'Docid', 'type': 'integer'}, 'text': {'title': 'Text', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'temporal_query_type': {'$ref': '#/$defs/TemporalQueryType'}, 'allen_relation': {'$ref': '#/$defs/AllenRelation'}}, 'required': ['docid', 'text', 'temporal', 'temporal_query_type', 'allen_relation'], 'title': 'Passage', 'type': 'object'}, 'TemporalQueryType': {'enum': ['Explicit', 'Implicit', 'TemporalAnswer'], 'title': 'TemporalQueryType', 'type': 'string'}}, 'properties': {'query_id': {'title': 'Query Id', 'type': 'integer'}, 'query': {'title': 'Query', 'type': 'string'}, 'temporal': {'items': {'type': 'string'}, 'title': 'Temporal', 'type': 'array'}, 'positive_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Positive Passages', 'type': 'array'}, 'negative_passages': {'items': {'$ref': '#/$defs/Passage'}, 'title': 'Negative Passages', 'type': 'array'}}, 'required': ['query_id', 'query', 'temporal', 'positive_passages', 'negative_passages'], 'title': 'TemporalAnnotation', 'type': 'object'}.

2. "query_id": must be assigned with the provided "query_id".

3. “query":
- A query is a provided sentence that may contain one or many temporal expressions.
- Please note that SUTIME only provides explicit temporal expressions for the query and they are not perfect; therefore, you must double-check them and further detect additional explicit and implicit temporal expressions such as events.
- You must extract all events that are anchored to specific datetime (e.g., “2000 FA Cup Final”, “the 2007 election”, etc.).
- The "temporal" field should be extracted as written from the query's text and they must be concise, such as: "in April, 1906", "2 July 2010", "2004 general election", etc.

4. "positive_passages":
- Natural, QA-style questions with diverse phrasing that seek information from the query. Each question must contain exactly one extractable temporal expression, logically align with one of the query's temporal expressions, yet be diverse in Allen relations.
- Based on the query's temporal expressions, you must generate "positive_passages" that cover all "TemporalQueryType", including 'Explicit', 'Implicit', and 'TemporalAnswer', when possible:
    - You MUST prioritise generating questions with explicit temporal constraints, like "in 2010", "from 2010 to 2015", etc. They must logically align with the query’s temporal expression(s), such as being equals, overlapping with, or being contained within the query’s temporal expression(s). If the query has multiple temporal expressions, the generated questions must logically align with at least one of them.
    - You should also prioritise generating questions with implicit temporal constraints, such as "after EVENT", "before EVENT". These must logically align with the query’s temporal expression(s).
    - For both explicit and implicit "TemporalQueryType" questions, you must not use phrases like what/which date/day/month/year/time or when etc., that inquire about time. You must not confuse this with TemporalAnswer questions.
    - You can also generate "TemporalAnswer" (asking for datetime/duration/time-range of an EVENT, e.g., "When did EVENT happen?", "What time did he arrive?"). For this type of question, you must not add any temporal expression and you must set: "TemporalQueryType": "TemporalAnswer", "allen_relation": "Empty", and "temporal": []. Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- "temporal" field: must extract all exact text spans of the temporal expressions that exist in the generated question (not normalized or paraphrased). Regarding explicit and implicit "TemporalQueryType", they must be concise temporal expressions as written, such as "after July 2010", "from 2012 to 2014", that are not just normalized dates. For instance, instead of "In 1906 he moved with his family to a farm", prefer the concise "In 1906" and keep prepositions or context words that anchor the time, e.g., 'from', 'in', 'after', etc. Regarding "TemporalAnswer" passages, the "temporal" field must be an empty list.
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- “docid”: must remain the same as provided.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality positive passages, prioritizing quality over quantity.

5. "negative_passages": follow the same format as "positive_passages", but represent contexts that are temporally mismatched or semantically irrelevant to the query. Based on the query's temporal expressions, you must generate "negative_passages" that cover all "TemporalQueryType", including "Explicit", "Implicit", and "TemporalAnswer", when possible. They must be hard, temporally-confused, yet diverse in Allen relations questions that cover all following cases:
- Case 1: Questions with temporal expressions that mismatch with the query’s temporal expression(s): 
    - If the query specifies a span (e.g., 2005–2007), use an interval that falls completely outside it (e.g., 2008, before 2005, after 2007).
    - Adjacent years or ranges are valid negatives (e.g., query = 2010, negative = 2009).
    - Use shifted but non-overlapping intervals (e.g., query = 2010, negative = 2012–2014).
    - Include misleading implicit cues (e.g., “shortly after 2011” vs. query “in 2010”).
    - You may reuse the same passage from "positive_passages" but replace its temporal expressions with mismatched ones.
- Case 2: Questions with same temporal expression but irrelevant event/entity
    - Questions with overlapping or identical temporal expressions but targeting a different subject.
    - Example: query = “Ronaldo’s career in 2010” vs. negative = “Messi’s career in 2010”.
    - This ensures negatives are temporally aligned but semantically irrelevant.
- Case 3: "TemporalAnswer"-type questions that are either:
    - Irrelevant to the query.
    - Or ask for non-existent temporal information in the query context.
    - Use sparingly (less than 2 per query, only when explicit/implicit cannot be formed).
- Allen relation: consider Passage = Event A and Query = Event B. Must be correct and align with the provided definition.
- Quantity: For each split/rewritten query, you must generate a list of 5 high-quality negative passages, prioritizing quality over quantity.  
- Do not create trivial negatives (e.g., completely unrelated random text).
- Ensure no accidental overlap with valid facts in the document.

6. "temporal_query_type": Must be "Explicit", "Implicit", or "TemporalAnswer". If the temporal contains a clear date/month/year, it is "Explicit", NOT "Implicit". If the passage asks for which date/month/year of an event, it is "TemporalAnswer".
   
7. Final output: Only output valid JSON(s). Do not explain, add comments, or include extra text, since your output will be parsed automatically.

### Demonstration 1
Input:
docid: Q2465
query_id: 0
query: "On 2 July 2010 , after helping Setúbal avoid top-flight relegation , Barbosa was released by Porto , signing a three-year contract with S.C . Braga."
SUTIME's output: [{'timex-value': '2010-07-02', 'start': 3, 'end': 14, 'text': '2 July 2010', 'type': 'DATE', 'value': '2010-07-02'}, {'timex-value': 'P3Y', 'start': 111, 'end': 121, 'text': 'three-year', 'type': 'DURATION', 'value': 'P3Y'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?.

Output:
{"query_id":0,"query":"On 2 July 2010, after helping Setúbal avoid top-flight relegation, Barbosa was released by Porto, signing a three-year contract with S.C. Braga.","temporal":["2 July 2010","three-year contract"],"positive_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2010 to 2013?","temporal":["from 2010 to 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for on 2 July 2010?","temporal":["2 July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa join after leaving Porto in July 2010?","temporal":["after leaving Porto in July 2010"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa sign a three-year contract with?","temporal":["three-year contract"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2465,"text":"After being released by Porto, which team did Hélder Barbosa sign a contract with?","temporal":["After being released by Porto"],"allen_relation":"MetBy","temporal_query_type":"Implicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between July 2010 and July 2013?","temporal":["between July 2010 and July 2013"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2465,"text":"When did Hélder Barbosa sign a contract with S.C. Braga.?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2465,"text":"How long did Hélder Barbosa's contract with S.C. Braga last?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2465,"text":"Which team did Hélder Barbosa play for from 2002 to 2009?","temporal":["from 2002 to 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for between 2006 and 2009?","temporal":["between 2006 and 2009"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for after 2014?","temporal":["after 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for during 2008?","temporal":["during 2008"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Hélder Barbosa play for prior to 2010?","temporal":["prior to 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2465,"text":"Which team did Ronaldo play for in July 2010?","temporal":["in July 2010"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 1

### Demonstration 2
Input:
docid: 2466
query_id: 1
query: "Rarely used in the first months , he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish ."
SUTIME's output: [{'timex-value': 'PXM', 'start': 15, 'end': 31, 'text': 'the first months', 'type': 'DURATION', 'value': 'PXM'}, {'timex-value': '2011-01', 'start': 78, 'end': 90, 'text': 'January 2011', 'type': 'DATE', 'value': '2011-01'}]
Example positive passages: Hélder Barbosa played for which team from 2010 to 2013?; Hélder Barbosa played for which team between June 2012 and October 2012?"

{"query_id":1,"query":"Rarely used in the first months, he began gaining more playing time after the January 2011 departure of Matheus , who left for a team in Ukraine , and contributed four league goals in an eventual fourth-place finish .","temporal":["after the January 2011 departure of Matheus"],"positive_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for between January 2011 and December 2011?","temporal":["between January 2011 and December 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for as of January 2011?","temporal":["as of January 2011"],"allen_relation":"Equals","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after Matheus departed in January 2011?","temporal":["after Matheus departed in January 2011"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa contribute goals to during the 2011 season?","temporal":["during the 2011 season"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for throughout 2011?","temporal":["throughout 2011"],"allen_relation":"During","temporal_query_type":"Explicit"},{"docid":2466,"text":"When did Hélder Barbosa start getting more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"},{"docid":2466,"text":"When did Hélder Barbosa begin gaining more playing time?","temporal":[],"allen_relation":"Empty","temporal_query_type":"TemporalAnswer"}],"negative_passages":[{"docid":2466,"text":"Which team did Hélder Barbosa play for before 2010?","temporal":["before 2010"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for after 2013?","temporal":["after 2013"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for in 2007?","temporal":["in 2007"],"allen_relation":"Before","temporal_query_type":"Explicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for prior to joining Braga?","temporal":["prior to joining Braga"],"allen_relation":"Before","temporal_query_type":"Implicit"},{"docid":2466,"text":"Which team did Hélder Barbosa play for during 2014?","temporal":["during 2014"],"allen_relation":"After","temporal_query_type":"Explicit"},{"docid":2466,"text":"After the January 2011 departure of Matheus, did Ronaldo get more playing time?","temporal":["After the January 2011 departure of Matheus"],"allen_relation":"Equals","temporal_query_type":"Explicit"}]}
### End of Demonstration 2"""

import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")

def normalize_and_split_string_by_punctuation(text, add_title=False):    
    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)
    
    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)    

    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"
    
    # Remove : and ;
    punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    
    if add_title:
        title = text.split(": ")[0]
        return [f"{title}: " +  x.strip() for ix, x in enumerate(re.findall(punct_regex, text)) if ix != 0]
    
    return [x.strip() for x in re.findall(punct_regex, text)]



In [ ]:
a = {"query_id":14659,"query":"Clemente Palma - Life: Upon his return to Peru , he resumed his position as curator of the National Library of Peru , a post that he held until November 1911 . During this period , he founded several cultural and literary magazines such as Prisma and Variedades and the daily newspaper La Crónica . From 1911 to 1918 , he dedicated himself to the direction of these magazines . He was director of the magazines Prisma ( 1906–1908 ) and Variedades ( 1908–1931 ) and the newspaper La Crónica ( 1912–1929 ) .","positive_passages":[{"docid":"Q2281130","text":"Clemente Palma was an employee for whom between Dec 1929 and 1930?"}]}

text = a["query"]
query_id = a["query_id"]
docid = int(a["positive_passages"][0]["docid"][1:])
positive_passages = a["positive_passages"]
# negative_passages = a["negative_passages"]

positive_passages = [x['text'] for x in positive_passages]
# negative_passages = [x['text'] for x in negative_passages]

query_list = normalize_and_split_string_by_punctuation(text,add_title=True)
final_query_list = []
final_sutime_list = []

messages = []

final_query_list = []
final_sutime_list = []

buffer = []

for q in query_list:
    parsed = sutime.parse(q)

    if len(parsed) == 0:
        # No temporal expression → keep merging
        buffer.append(q)
    else:
        # Check if merging with buffer creates more temporal expressions
        if buffer:
            merged = " ".join(buffer + [q])
            if len(sutime.parse(merged)) > 1:
                temp = " ".join(buffer)
                final_query_list.append(temp)
                final_sutime_list.append(sutime.parse(temp))
                buffer = [q]
                continue
        buffer.append(q)

# Flush leftover
if buffer:
    merged = " ".join(buffer)
    temp = sutime.parse(merged)
    if len(temp) == 1:
        final_query_list.append(merged)
        final_sutime_list.append(temp)

for q, sutime_output in zip(final_query_list, final_sutime_list):    
    if not (len(sutime_output) > 0 and len(word_count_regex.findall(q)) > 5):
        continue

    for t in sutime_output:
        s, e = t["start"], t["end"]
        t["value"] = q[s:e]

    content = f"Input:\ndocid: {docid}\nquery_id: {query_id}\nquery: {q}\nSUTIME's output: {sutime_output}\nExample positive passages: {','.join(positive_passages)}\nOutput:"
    
    messages.append([
        {"role":"system", "content":prompt_template},
        {"role":"user", "content":{content},}
    ])

In [688]:
messages

[[{'role': 'system',
   'content': 'You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME\'s output for your reference. We also provide you with previously annotated positive samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.\nYour final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.\nYour temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. \nYou must follow the definitions and instructions below.\n\n* Allen relations and descriptions:\n- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?\n- After: Did ‘Ev

In [689]:
output = llm.chat(
    messages=messages, 
    sampling_params=sampling_params,
    # chat_template_kwargs={"enable_thinking": True},
)

Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

In [ ]:
temporal_answer_list = [
    # Point in time
    "what day",
    "what time",
    "what date",
    "what month",
    "what year",
    "which day",
    "which time",
    "which date",
    "which month",
    "which year",
    "when did",
    "when was",
    "when were",
    "at what time",
    "at what date",
    "at what year",
    "on what day",
    "on what date",
    "on what year",

    # Duration / span
    "how long",
    "how many days",
    "how many weeks",
    "how many months",
    "how many years",
    "for how long",
    "over what period",
    "during what years",
    "during which year",
    "for what duration",
    "in what year range",
    "between what years",

    # Frequency / recurrence
    "how often",
    "how frequently",

    # Relative temporals
    "since when",
    "until when",
    "from when",
    "from what year",
    "from what date",
    "to what year",
    "to what date",
    "up to when",
    "as of when",
    "by what year",
    "by when",
    "around when",
    "at what age",
]


def fix_implicit_temporal(passage, sutime):
    """
    Adjust temporal_query_type if 'implicit' but tagger detects explicit datetime.
    """
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Implicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    if len(sutime.parse(passage["temporal"][0].lower())) > 0:
        passage["temporal_query_type"] = TemporalQueryType.Explicit
        
    lower_passage = passage["text"].lower()
    
    # Make sure that the
    implicit_temporal = []
    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            implicit_temporal.append(passage["text"][start:start+len(t)])    
    
    if not implicit_temporal:
        return {}
    
    passage["temporal"] = implicit_temporal
    return passage


def fix_temporal_answer(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.TemporalAnswer:
        return passage

    lower_passage = passage['text'].lower()
    if any(word in lower_passage for word in temporal_answer_list):
        for word in temporal_answer_list:
            if word in lower_passage:
                lower_passage = lower_passage.replace(word, "")
        
    # Ensure that it passes the SUTIME tagger.
    if len(sutime.parse(lower_passage)) > 0:
        return {}
    else:
        passage['temporal'] = []
        passage['allen_relation'] = AllenRelation.Empty
    # print(passage)
    return passage


def fix_explicit_temporal(passage, sutime):
    if passage == {} or passage["temporal_query_type"] != TemporalQueryType.Explicit:
        return passage
    
    # Must have exactly 1 temporal
    if len(passage["temporal"]) != 1:
        return {}
    
    lower_passage = passage["text"].lower()
    temporal = []

    for t in passage["temporal"]:
        start = lower_passage.find(t.lower())
        if start != -1:
            temporal.append(passage["text"][start:start+len(t)])

    if not temporal:
        return {}

    passage["temporal"] = temporal
    return passage


def validate_passages(passages, sutime):
    """
    Validate and filter passages.
    """
    valid = []
    
    for p in passages:
        p = fix_implicit_temporal(p, sutime)
        p = fix_explicit_temporal(p, sutime)
        p = fix_temporal_answer(p, sutime)
        
        if p:
            valid.append(p)
    return valid


def validate_temporal(js, max_temporal_expressions=3):
    lower_query = js["query"].lower()
    temporal = []
    
    # Ensure that the temporal is extracted as written from the original query
    for t in js["temporal"]:
        start = lower_query.find(t.lower())
        if start != -1:
            temporal.append(js["query"][start:start+len(t)])

    if len(temporal) == 0 or len(temporal) > max_temporal_expressions:
        return []
    return temporal



In [691]:
temporal_jsonl = []
temp = []
for sample in output:
    sample = sample.outputs[0].text # Get the generated text
    list_of_raw_js = []
    query_set = set()
    
    for x in sample.split("\n"): # Split by newline to get multiple JSONs if any
        parts = x.split(',{"query_id"')
        if len(parts) > 0:
            json_strings = [parts[0]] + [',{"query_id"' + p for p in parts[1:]]
            list_of_raw_js.extend(json_strings)
        else:
            list_of_raw_js.append(x)
    
    for raw_js in list_of_raw_js:
        raw_js = raw_js.strip()
        
        if raw_js == "":
            continue
        try:
            js = TemporalAnnotation.model_validate_json(repair_json(raw_js, ensure_ascii=False)).model_dump()
            
            if len(js["temporal"]) == 0 or len(js["positive_passages"]) == 0 or len(js["negative_passages"]) == 0 or js["query"] in query_set:
                continue
            else:
                print(js["query"])
                js["temporal"] = validate_temporal(js)
                
                if len(js["temporal"]) == 0:
                    continue
                
                # Validate passages
                js["positive_passages"] = validate_passages(
                    js["positive_passages"], sutime
                )
                
                if len(js["positive_passages"]) == 0:
                    continue
                
                js["negative_passages"] = validate_passages(
                    js["negative_passages"], sutime
                )
                
                if len(js["negative_passages"]) == 0:
                    continue
                
                temp.append(js)
                query_set.add(js["query"])
                # positive = defaultdict(int)
                # negative = defaultdict(int)
                # allen_relation = defaultdict(int)

                # for j in js["positive_passages"]:
                #     positive[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                # for j in js["negative_passages"]:
                #     negative[j["temporal_query_type"]] += 1
                #     allen_relation[j["allen_relation"]] += 1
                
                # print(positive)
                # pprint(positive)
                # print(positive)
                # pprint(negative)

        except Exception as e:
            print("Error:", e)
            print("Offending JSON:", raw_js)
            continue
temporal_jsonl.extend(temp)

Clemente Palma - Life: During this period, he founded several cultural and literary magazines such as Prisma and Variedades and the daily newspaper La Crónica.
Clemente Palma - Life: From 1911 to 1918, he dedicated himself to the direction of these magazines.


In [692]:
messages

[[{'role': 'system',
   'content': 'You are a temporal annotation expert for information retrieval. We provide you with a query, split from a long document, that may contain one or many temporal expressions and the corresponding SUTIME\'s output for your reference. We also provide you with previously annotated positive samples for the original long document; you may reuse them or use them as references only if they are semantically relevant to the current query. Your job is to generate high-quality positive passages and negative passages for temporal contrastive learning.\nYour final output are annotated JSONs (no indentation) that strictly follow the provided JSON schema.\nYour temporal annotations must be more precise and reliable than SUTIME or HeidelTime, with top-tier accuracy. \nYou must follow the definitions and instructions below.\n\n* Allen relations and descriptions:\n- Before: Did ‘Event A’ occur before ‘Event B’ without any overlap between the two events?\n- After: Did ‘Ev

In [693]:
temporal_jsonl

[{'query_id': 14659,
  'query': 'Clemente Palma - Life: During this period, he founded several cultural and literary magazines such as Prisma and Variedades and the daily newspaper La Crónica.',
  'temporal': ['daily'],
  'positive_passages': [{'docid': 2281130,
    'text': 'During which period did Clemente Palma found Prisma and Variedades?',
    'temporal': [],
    'temporal_query_type': <TemporalQueryType.TemporalAnswer: 'TemporalAnswer'>,
    'allen_relation': <AllenRelation.Empty: 'Empty'>},
   {'docid': 2281130,
    'text': 'Which cultural magazine did Clemente Palma found during this period?',
    'temporal': ['during this period'],
    'temporal_query_type': <TemporalQueryType.Implicit: 'Implicit'>,
    'allen_relation': <AllenRelation.During: 'During'>},
   {'docid': 2281130,
    'text': 'When was La Crónica founded by Clemente Palma?',
    'temporal': [],
    'temporal_query_type': <TemporalQueryType.TemporalAnswer: 'TemporalAnswer'>,
    'allen_relation': <AllenRelation.Empt

In [698]:
del llm
cleanup_dist_env_and_memory()

## Post-processing

In [90]:
temp_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/merged_train.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 7042 items.


In [91]:
# temporal_jsonl
positive = defaultdict(int)
negative = defaultdict(int)
allen_relation = defaultdict(int)
for x in temp_jsonl:
    for i in x["positive_passages"]:
        positive[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
    for i in x["negative_passages"]:
        negative[i["temporal_query_type"]] += 1
        allen_relation[i["allen_relation"]] += 1
pprint(positive)
pprint(negative)

defaultdict(<class 'int'>,
            {'Explicit': 27515,
             'Implicit': 983,
             'TemporalAnswer': 8646})
defaultdict(<class 'int'>,
            {'Explicit': 34823,
             'Implicit': 465,
             'TemporalAnswer': 492})


### Check for limited positive/negative passages

In [92]:
count = 0
new_temp_jsonl = []
for x in temp_jsonl:
    if len(x["positive_passages"]) <= 1:
        print("pos", x["query_id"])
        count += 1
        continue
    if len(x["negative_passages"]) <= 1:
        print("nega", x["query_id"])
        count += 1
        continue
    new_temp_jsonl.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)
print(count)

nega 298
pos 368
pos 620
pos 986
pos 1233
pos 9111
nega 3713
pos 4105
pos 4721
pos 6222
10


### Check for positive/negative passages that only contain a reference year (2025)

In [93]:
new_temp_jsonl2 = []
count = 0
for x in new_temp_jsonl:
    t = ["2025" in j["text"] for j in x["positive_passages"]]
    if all(t):
        print(x["query_id"])
        print(t)
        count += 1
    else:
        new_temp_jsonl2.append(x)
print(count)   

733
[True, True, True, True]
2681
[True, True, True, True]
3368
[True, True, True, True]
7074
[True, True, True]
4


### Ensure that either positive and negative passages must contain at least one explicit/implicit temporal

In [97]:
count_pos = 0
count_neg = 0
count_both = 0
count = 0
query_set = set()
new_temp_jsonl3 = []

for x in new_temp_jsonl2:
    check_pos = False

    for i in x["positive_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_pos = True
            break 

    if check_pos == False:
        # print("pos", x["query_id"], x["query"][:10])
        count_pos += 1
        query_set.add(x["query_id"])

    check_neg = False    
    for i in x["negative_passages"]:
        if i["temporal_query_type"] == "Implicit" or i["temporal_query_type"] == "Explicit":
            check_neg = True
            break

    if check_neg == False:
        count_neg += 1
        # print("neg", x["query_id"], x["query"][:10])
        query_set.add(x["query_id"])
        
    if check_neg == True or check_pos == True:
        query_set.add(x["query_id"])
        new_temp_jsonl3.append(x)
    
    if not check_neg and not check_pos:
        count_both += 1
        print(x["query_id"], x["query"][:10])
    
print(count, count_pos, count_neg, count_both)

621 Sergey Sir
649 Sean Bean 
967 ACES Colom
9143 The One Sh
9152 Clemente P
3822 Michael St
4760 David Fost
5257 Richard J.
5766 Taeyeon - 
5796 Keystone C
5798 Keystone C
5813 Ralph Hart
6140 Kenneth O.
6261 Stanley Ho
6782 Lamar Univ
7131 Reginald M
7275 Olivia Hus
7811 Arthur Ric
7861 Max Baucus
8555 William Pi
8623 Mike Penni
0 48 28 21


In [99]:
len(new_temp_jsonl3)

7007

### Check for relative temporal expressions in the queries

In [103]:
count = 0
new_temp_jsonl4 = []

for x in new_temp_jsonl3:
    check = False

    if "now" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "now")
        count += 1
        check = True
        # break
    if "today" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "today")
        count += 1
        check = True
        # break
    if "current" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "current")
        count += 1 
        check = True
        # break
    if "yesterday" in x["temporal"]: # or "today" in x["temporal"] or "current" in x["temporal"]:
        print(x["query_id"], "yesterday")
        count += 1     
        check = True
        # break    
    if not check:
        new_temp_jsonl4.append(x)
# write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl", new_temp_jsonl, jsonl=True)
print(count)

79 today
105 today
276 current
279 now
313 now
411 now
499 now
684 now
743 current
957 now
1138 now
2593 now
2674 now
2687 today
2715 now
2845 now
2854 now
2894 now
2962 today
2964 now
2979 now
3272 now
3297 today
3304 today
3340 now
3390 current
3705 current
3734 now
3812 current
4047 now
4167 now
4187 now
4220 now
4234 now
4355 current
4504 now
4505 current
4649 today
4747 current
4828 now
4854 now
5224 today
5374 now
5610 current
5891 now
6067 now
6438 now
6583 now
6657 now
6701 now
6946 now
6949 now
7203 now
7331 now
7353 now
7404 now
7465 now
7720 today
7770 current
7911 now
8045 now
8045 current
8045 now
8108 now
8292 now
8318 now
8331 today
8354 now
8488 current
8570 now
8676 now
8761 now
8902 now
8904 today
74


In [106]:
write_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/train.jsonl", new_temp_jsonl4, jsonl=True)

The file contains 6933 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/train.jsonl


## Generate the test folder

In [6]:
import re

# Precompile regexes for speed if this runs many times
_whitespace_re = re.compile(r"\s+")
_punct_spacing_re = re.compile(r"\s*([,.;:?!])\s*")
_char_space_re = re.compile(r"\b(?:[A-Za-z]\s+)+[A-Za-z]\b")
word_count_regex = re.compile(r"\w+")
# _sentence_re = re.compile(r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?")


def search_paragraphs_index(
    recon_start_end_list, answer_start_end, recon_context, targets, paragraphs
):
    """Return the index of the paragraph span that contains the answer span."""
    potential_ix = []
    for ix, (s, e) in enumerate(recon_start_end_list):
        if targets in paragraphs[ix]["text"]:
            potential_ix.append(ix)
        if (
            answer_start_end[0] >= s
            and answer_start_end[1] <= e
            and targets in paragraphs[ix]["text"]
        ):
            return ix

    potential_start_end_list = [recon_start_end_list[x] for x in potential_ix]
    for ix, (s, e) in enumerate(potential_start_end_list):
        if answer_start_end[0] >= s:
            return ix
    return -1


def format_paragraph(paragraphs, art_title, idx, dedup_titles):
    par_title = paragraphs[idx]["title"]
    if dedup_titles and art_title == par_title:
        title_str = art_title
    else:
        title_str = f"{art_title} - {par_title}"
    return f"{title_str}: {paragraphs[idx]['text']}"


def extract_paragraphs_with_answers(
    paragraphs, answer_starts, answer_ends, targets, dedup_titles=True
):
    """
    Given Wikipedia-style paragraphs and answer spans (start/end indices),
    return a list of disambiguated paragraph strings that include both
    article title and section title.

    Parameters
    ----------
    paragraphs : list[dict]
        Each dict must have keys: "title", "text".
    answer_starts : list[int]
        List of answer start indices (aligned with reconstructed context).
    answer_ends : list[int]
        List of answer end indices.
    dedup_titles : bool, default=True
        If True, collapses "Title - Title" into "Title".

    Returns
    -------
    list[str] : formatted paragraphs with titles and text
    """

    title_set = set()
    start_end_list = []
    recon_context = ""
    final_paragraph_list = []

    # Build context + paragraph spans
    for par in paragraphs:
        title = par["title"]
        text = par["text"].strip()

        # Title case
        if len(title_set) == 0:
            recon_context += title.strip()
            title_set.add(title)

        # Add article/section title once
        if title not in title_set:
            recon_context += " " + title.strip() + " . "
            title_set.add(title)

        start_idx = len(recon_context)
        recon_context += " " + text.strip() + " "
        end_idx = len(recon_context)

        start_end_list.append((start_idx, end_idx))
    recon_context = recon_context.replace("  ", " ")

    # Match answers back to paragraph(s)
    for t, s, e in zip(targets, answer_starts, answer_ends):
        ix = search_paragraphs_index(
            start_end_list, (s, e), recon_context, t, paragraphs
        )
        art_title = paragraphs[0]["title"]

        if ix != -1:
            final_paragraph_list.append(
                format_paragraph(paragraphs, art_title, ix, dedup_titles)
            )
            continue

        # fallback: search in recon_context
        t_lower = t.lower()
        for ix, (_, end) in enumerate(start_end_list):
            if t_lower in recon_context[:end].lower():
                nearby_idxs = range(max(0, ix - 1), min(len(paragraphs), ix + 1))
                combined = " ".join(
                    format_paragraph(paragraphs, art_title, j, dedup_titles)
                    for j in nearby_idxs
                )
                final_paragraph_list.append(combined)
                break

    return final_paragraph_list


def normalize_and_split_string_by_punctuation(text):

    text = re.sub(r"[^\w\s]", "", text)

    # Normalize whitespace
    text = _whitespace_re.sub(" ", text.strip())

    # Fix spacing around punctuation (any of ,.;:?!)
    text = _punct_spacing_re.sub(r"\1 ", text)

    # Fix character-level split words
    text = _char_space_re.sub(lambda m: m.group(0).replace(" ", ""), text)

    return text

    # # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!;:]|\.(?!\s+[A-Z]))*.?"

    # # Remove : and ;
    # punct_regex = r"(?=\S)(?:[A-Z][a-z]{0,3}\.|[^.?!]|\.(?!\s+[A-Z]))*.?"
    # return [x.strip() for x in re.findall(punct_regex, text)]

In [7]:
df_test = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test.parquet"
)
corpus_parquet = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/corpus.parquet"
)
corpus_parquet["text"] = corpus_parquet["text"].apply(
    lambda x: normalize_and_split_string_by_punctuation(x).lower()
)

In [9]:
corpus_parquet

,index,id,title,text
0,824,Q30122825,Doug Jones (politician),doug jones politician gordon douglas jones bor...
1,825,Q30122825,Doug Jones (politician),his father worked at us steel and his mother w...
2,826,Q30122825,Doug Jones (politician),blanton was up for parole in 2016 jones spoke ...
3,827,Q30122825,Doug Jones (politician),jones was sworn in on january 3 2018 alongside...
4,828,Q30122825,Doug Jones (politician),us representative bradley byrne was also a con...
...,...,...,...,...
22448,609334,Q3661950,Cassa di Risparmio della Marca Trivigiana,cassa di risparmio della marca trivigiana cass...
22449,609335,Q3661950,Cassa di Risparmio della Marca Trivigiana,after the unification of italy in 1866 the sav...
22450,609336,Q3661950,Cassa di Risparmio della Marca Trivigiana,a further amendment of laws in 1994 making dir...
22451,609337,Q3661950,Cassa di Risparmio della Marca Trivigiana,the former direct owner of cassamarca spa the ...


In [8]:
df_test

,idx,question,context,targets,paragraphs,wiki_id,unanswerable
0,/wiki/Attaphol_Buspakom#P54#0,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Port F.C, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
1,/wiki/Attaphol_Buspakom#P54#1,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Pahang FA, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
2,/wiki/Attaphol_Buspakom#P54#2,Which team did Attaphol Buspakom play for in J...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Port F.C, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
3,/wiki/Attaphol_Buspakom#P54#3,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Pahang FA, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
4,/wiki/Attaphol_Buspakom#P54#4,Which team did Attaphol Buspakom play for afte...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...",[Thailand national football team],"[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
...,...,...,...,...,...,...,...
3073,/wiki/Jörg_Widmann#P463#0,Jörg Widmann became a member of what organizat...,Jörg Widmann Jörg Widmann ( born 19 June 1973 ...,[],[{'text': ' Jörg Widmann ( born 19 June 1973 )...,Q462746,True
3074,/wiki/Jörg_Widmann#P463#1,Jörg Widmann became a member of what organizat...,Jörg Widmann Jörg Widmann ( born 19 June 1973 ...,[Akademie der Wissenschaften und der Literatur],[{'text': ' Jörg Widmann ( born 19 June 1973 )...,Q462746,False
3075,/wiki/Jörg_Widmann#P463#2,Jörg Widmann became a member of what organizat...,Jörg Widmann Jörg Widmann ( born 19 June 1973 ...,[Freie Akademie der Künste Hamburg],[{'text': ' Jörg Widmann ( born 19 June 1973 )...,Q462746,False
3076,/wiki/Jörg_Widmann#P463#3,Jörg Widmann became a member of what organizat...,Jörg Widmann Jörg Widmann ( born 19 June 1973 ...,[Deutsche Akademie der Darstellenden Künste],[{'text': ' Jörg Widmann ( born 19 June 1973 )...,Q462746,False


In [25]:
output_jsonl = []
count = 0
count_unanswerable = 0
count0 = set()
test_jsonl = []

df_qrel = []

for i in range(len(df_test)):
    sample = df_test.iloc[i]

    targets = sample['targets'].tolist()
    wiki_id = sample["wiki_id"]

    corpus_rows = corpus_parquet[(corpus_parquet["id"] == wiki_id)]

    if sample["unanswerable"]:
        # df_qrel.append((i, tuple(corpus_rows.index.tolist())))
        count_unanswerable += 1
        continue

    for t in targets:
        t = normalize_and_split_string_by_punctuation(t).lower()

        index_list = corpus_rows[corpus_rows["text"].str.contains(t)].index

        if len(index_list) >= 1:
            df_qrel.append((i, tuple(index_list.tolist())))
        if len(index_list) == 0:
            print(targets)
            print(corpus_rows.head())
            count += 1
            continue
    test_jsonl.append(
        {
            "query_id": i,
            "query": sample["question"],
            "answers": sample["targets"].tolist(),
            "docid": wiki_id
        }
    )
    # count += 1
    #     count.add(wiki_id)
    # if len(index_list) == 0:
    #     count0.add(wiki_id)
    #     # print(t)
    #     # print(" ".join(corpus_rows["text"].values))
    #     # break

['Olympique Lyonnais']
      index        id            title  \
523  412978  Q2497888  Olivier Bernard   
524  412979  Q2497888  Olivier Bernard   
525  412980  Q2497888  Olivier Bernard   

                                                  text  
523  olivier bernard olivier bernard born 14 octobe...  
524  after just one season with rangers and only ni...  
525  ill do everything i can to get players to durh...  
['Eva Schubach']
        index     id             title  \
12535  635603  Q2530  Gerhard Schröder   
12536  635604  Q2530  Gerhard Schröder   
12537  635605  Q2530  Gerhard Schröder   
12538  635606  Q2530  Gerhard Schröder   
12539  635607  Q2530  Gerhard Schröder   

                                                    text  
12535  gerhard schröder gerhard fritz kurt schröder b...  
12536  among his more controversial cases schröder he...  
12537  between 1994 and 1998 he was also chairman of ...  
12538  titled europe the third way or die neue mitte ...  
12539  in respo

In [38]:
corpus_parquet[corpus_parquet["id"] == "Q2530"]["text"]

12535    gerhard schröder gerhard fritz kurt schröder b...
12536    among his more controversial cases schröder he...
12537    between 1994 and 1998 he was also chairman of ...
12538    titled europe the third way or die neue mitte ...
12539    in response a grouping of leftwing spd disside...
12540    most voters soon associated schröder with the ...
12541    in his first months in office schröder vigorou...
12542    marking a clear break with the caution of germ...
12543    schröder represented the german government at ...
12544    he also sought to detach himself from the clos...
12545    after leaving public office schröder represent...
12546    in an editorial entitled gerhard schroeders se...
12547    schröder has criticised some european countrie...
12548    schröders fifth marriage has earned him the ni...
Name: text, dtype: object

In [19]:
len(test_jsonl), len(df_test), count, count_unanswerable

(2613, 3078, 188, 465)

In [20]:
temp = pd.DataFrame(test_jsonl)

In [34]:
df_test.head()

,idx,question,context,targets,paragraphs,wiki_id,unanswerable
0,/wiki/Attaphol_Buspakom#P54#0,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Port F.C, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
1,/wiki/Attaphol_Buspakom#P54#1,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Pahang FA, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
2,/wiki/Attaphol_Buspakom#P54#2,Which team did Attaphol Buspakom play for in J...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Port F.C, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
3,/wiki/Attaphol_Buspakom#P54#3,Which team did Attaphol Buspakom play for betw...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...","[Pahang FA, Thailand national football team]","[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False
4,/wiki/Attaphol_Buspakom#P54#4,Which team did Attaphol Buspakom play for afte...,"Attaphol Buspakom Attaphol Buspakom ( ; ) , ni...",[Thailand national football team],"[{'text': ' Attaphol Buspakom ( ; ) , nickname...",Q757903,False


In [40]:
print(" ".join(df_test[df_test["wiki_id"] == "Q2530"]["context"]))

Gerhard Schröder Gerhard Fritz Kurt Schröder ( ; born 7 April 1944 ) is a German politician who served as Chancellor of Germany from 1998 to 2005 , during which his most important political initiative was Agenda 2010 . As the Leader of the Social Democratic Party of Germany ( SPD ) , he led a coalition government of the SPD and the Greens . Schröder has been chairman of Russian energy company Rosneft since 2017 . Before becoming a full-time politician , he was a lawyer , and before becoming Chancellor he served as Prime Minister of Lower Saxony ( 1990–1998 ) . Following the 2005 federal election , which his party lost , after three weeks of negotiations he stood down as Chancellor in favour of Angela Merkel of the rival Christian Democratic Union . He is currently the chairman of the board of Nord Stream AG and of Rosneft , after having been hired as a global manager by investment bank Rothschild , and also the chairman of the board of football club Hannover 96 . Early life and educati

In [41]:
print(" ".join(corpus_parquet[corpus_parquet["id"] == "Q2530"]["text"]))

gerhard schröder gerhard fritz kurt schröder born 7 april 1944 is a german retired politician lawyer consultant and lobbyist who served as the chancellor of germany from 1998 to 2005 during which his most important political initiative was agenda 2010 as chancellor he led a coalition government of his social democratic party of germany and the alliance 90the greens from 1999 to 2004 he was the leader of the social democratic party of germany spd schröder has been chairman of russian energy company rosneft since 2017 before becoming a fulltime politician he was a lawyer and before becoming chancellor he served as minister president of lower saxony 19901998 following the 2005 federal election which his party lost after three weeks of negotiations he stood down as chancellor in favour of angela merkel of the rival christian democratic union he is currently the chairman of the board of nord stream ag and of rosneft after having been hired as a global manager by investment bank rothschild a

In [312]:
with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/qrel.txt",
    "w",
) as outfile:
    for ix, v in enumerate(df_qrel):
        query_id = v[0]

        for docid in v[1]:
            outfile.write(f"{query_id} 0 {docid} {1}\n")

with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/qrel.txt",
    "r",
) as f:
    temp = [x for x in f.readlines()]

# from natsort import natsorted
# temp = natsorted(list(set(temp)), key=lambda x: x.split(" ")[0])

with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/qrel.txt",
    "w",
) as outfile:
    for i in set(temp):
        outfile.write(i)

In [ ]:
# corpus_parquet = pd.read_parquet(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/corpus.parquet"
# )
# corpus_jsonl = []
# for i in range(len(corpus_parquet)):
#     row = corpus_parquet.iloc[i]
#     corpus_jsonl.append({"docid": i, "text": row["text"]})


In [ ]:
# write_json(
#     "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/corpus.jsonl", corpus_jsonl, jsonl=True
# )

The file contains 22453 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/corpus.jsonl


In [316]:
write_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/query.jsonl", test_jsonl, jsonl=True
)

The file contains 2613 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/test/query.jsonl


### Prepare the dev

In [176]:
df_dev = pd.read_parquet(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/dev.parquet"
)

In [185]:
dev_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/hard/dev.hard.json",
    jsonl=True,
)

wiki_url_regex = re.compile(r"([^#]+)")


def process(x):
    return mapper.url_to_id(re.sub(r"amp;", "", wiki_url_regex.match(x["idx"])[0]))


train_id = []
dev_id = []
test_id = []

if __name__ == "__main__":
    # with Pool(16) as pool:  # use all available cores
    #     train_id = list(tqdm(pool.imap(process, train_jsonl), total=len(train_jsonl)))

    with Pool(16) as pool:  # use all available cores
        dev_id = list(tqdm(pool.imap(process, dev_jsonl), total=len(dev_jsonl)))

    # with Pool(16) as pool:  # use all available cores
    #     test_id = list(tqdm(pool.imap(process, test_jsonl), total=len(test_jsonl)))

The file is of type: <class 'list'>
The file contains 3087 items.


  0%|          | 0/3087 [00:00<?, ?it/s]

In [187]:
train_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/explitcit/time-sensitive-qa/hard/train.hard.json", jsonl=True
)

The file is of type: <class 'list'>
The file contains 14681 items.


In [188]:
train_jsonl[0]

{'idx': '/wiki/Knox_Cunningham#P39#0',
 'question': 'Which position did Knox Cunningham hold before Apr 1956?',
 'context': 'Knox Cunningham Sir Samuel Knox Cunningham , 1st Baronet , QC ( 3 April 1909 – 29 July 1976 ) , was a Northern Irish barrister , businessman and politician . As an Ulster Unionist politician at a time when the Unionists were part of the Conservative Party , he was also a significant figure in United Kingdom politics as Parliamentary Private Secretary to Harold Macmillan . His nephew was Sir Josias Cunningham . Early career . Cunningham was from an Ulster family . His father was Samuel Cunningham , and his mother was Janet Muir Knox ( nee McCosh ) of Dalry , Ayrshire . His elder brothers were Colonel James Glencairn Cunningham , Josias Cunningham stockbroker , Dunlop McCosh Cunningham owner of Murrays tobacco works , Belfast . He was sent to the Royal Belfast Academical Institution , and then to Fettes College in Edinburgh . He then won a place at Clare College , 

In [ ]:
output_jsonl = []
count = 0

for i in range(len(df_dev)):
    sample = df_dev.iloc[i]

    if sample["unanswerable"]:
        continue

    targets = sample["targets"].tolist()
    # start, end = sample["from"].tolist(), sample["end"].tolist()
    paragraphs = sample["paragraphs"].tolist()
    # print(len(sample['context']), sample['context'])

    result = extract_paragraphs_with_answers(paragraphs, start, end, targets)

    if len(result) == 0:
        # Final resort
        temp = normalize_and_split_string_by_punctuation(sample["context"])
        for ix, chunk in enumerate(temp):
            if targets[0] in chunk:
                start = max(0, ix - 5)
                end = min(len(temp), ix + 5)
                result = temp[start:end]
                break
        if len(result) == 0:
            print(i, sample["targets"][0], sample["targets"][0] in sample["context"])
            count += 1

    for res in result:
        output_jsonl.append(
            {
                "query_id": i,
                "query": res,
                "positive_passages": [
                    {"docid": sample["wiki_id"], "text": sample["question"]}
                ],
            }
        )

KeyError: 'from'

In [ ]:
count = 0
output_jsonl = []

import collections
output_dict = defaultdict(list)

for ix, i in enumerate(dev_jsonl):
    row = i
    # print(row["targets"])

    targets = [normalize_and_split_string_by_punctuation(x) for x in row["targets"]]

    paragraph_dict = collections.defaultdict(str)

    for par in row["paragraphs"]:
        paragraph_dict[par["title"]] += " " + par["text"] + " "

    paragraph_dict = {
        k: normalize_and_split_string_by_punctuation(v) for k, v in paragraph_dict.items()
    }

    paragraphs = [k+" "+v for k,v in paragraph_dict.items()]

    for iy, t in enumerate(targets):
        check = False
        for iz, par in enumerate(paragraphs):
            if t in par:
                output_dict[par].append(row["question"])
                # output_jsonl.append({
                #     "query_id": count,
                #     "query": par,
                #     "positive_passages":[{"text":row["question"]}],
                # })
                count += 1
                # check = True
                
        # if check == False:
        #     count += 1
        #     print(ix, iy)

In [221]:
len(dev_jsonl), len(output_jsonl)

(3087, 8012)

# Process for TempRetriever

In [317]:
train_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/enhanced_temporal/v4/train.jsonl",
    jsonl=True,
)

The file is of type: <class 'list'>
The file contains 11693 items.


In [318]:
new_train_jsonl = [
    {
        "query_id": x["query_id"],
        "query": x["query"],
        "positive_passages": [i for i in x["positive_passages"] if i["temporal_query_type"] != "TemporalAnswer"],
        "negative_passages": [i for i in x["negative_passages"] if i["temporal_query_type"] != "TemporalAnswer"]
    }
    for x in train_jsonl
]

In [322]:
new_train_jsonl2 = []
for i in new_train_jsonl:
    if len(i["positive_passages"]) == 0:
        continue 
    if len(i["negative_passages"]) == 0:
        continue 
    new_train_jsonl2.append(i)

# Dev

In [32]:
with open(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/qrel.txt",
    "r",
) as f:
    lines = f.readlines()

qid_list = [int(x.split()[0]) for x in lines]

In [33]:
query_jsonl = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/query.jsonl", jsonl=True
)

The file is of type: <class 'list'>
The file contains 2613 items.


In [34]:
new_query_jsonl = [
    x for x in query_jsonl if x["query_id"] in qid_list
]

In [35]:
qid_list[:10]

[2919, 1406, 1254, 1659, 684, 286, 117, 288, 218, 1334]

In [36]:
len(new_query_jsonl)

2439

In [37]:
id_set1 = set([x["query_id"] for x in new_query_jsonl])
id_set2 = set(qid_list)